# 04 - Bit-accurate check of a classifier against its fixed-point hardware version

The FPGA does not run the floating-point network. It runs the hls4ml C++ version with fixed-point types, and the decisions
can differ near the threshold. This notebook compiles that C++ (the same `NN.cpp`, `nnet_utils` and `ap_fixed` types that
synthesis uses) into a library and runs it on replayed traces next to the floating-point model. That gives:

- the **quantization cost**: fidelity of the fixed-point model minus the float model;
- the **decision agreement** between the two, shot by shot;
- the fixed-point probabilities, which the hardware should reproduce **bit for bit**. That is how the paper shows
  NN_live and NN_axi are bit-exact on every recorded shot (paper Sec. VII–VIII).

**You need:** a trained model (`.h5`, from notebook 03 or one of the deployed models in `models/`), replayed
traces with labels (a capture from notebook 02, or one listed in `DATA_MANIFEST.csv`), hls4ml, and a C++ compiler
(`g++`). On Windows, hls4ml's compile step needs MSYS2's g++; the optional cell below sets that up.

In [ ]:
from pathlib import Path

DATA_DIR = Path("..")
MODEL_H5 = DATA_DIR / "models/model_signed_domain_seed20260901_w576_box.h5"   # deployed NN_live, boxcar path
WINDOW = 576                                   # the model's window (its input is 2*WINDOW values)
EVAL = DATA_DIR / "loopback20k_box_A.npz"      # replayed traces: an .npz with arrays (KEY, "y"), or an .npy with <name>_y.npy
EVAL_KEY = "box"                               # for an .npz: "box" (boxcar path) or "fir" (QICK's FIR path)
N_EVAL = 2000                                  # shots to score (all 20,000 take a few minutes)
OUT_DIR = Path("outputs/fixed_point")
MSYS_BIN = Path(r"C:\msys64\ucrt64\bin")       # Windows only: where MSYS2's g++ lives

OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["TF_USE_LEGACY_KERAS"] = "1"        # QKeras needs the legacy Keras API
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import matplotlib.pyplot as plt
import numpy as np
from qkeras.utils import load_qmodel

model = load_qmodel(str(MODEL_H5), compile=False)
model.summary()

## 1. Replayed traces, in the firmware's input format

Each ADC word becomes a signed 16-bit integer, exactly as the firmware reads it. Keep the first `WINDOW` samples.

In [ ]:
def hw_input(X, window):
    Xi = np.clip(np.round(np.asarray(X, dtype=np.float64)), -32768, 32767).astype(np.int16).astype(np.float32)
    return np.ascontiguousarray(Xi[:, :2 * window])


if str(EVAL).endswith(".npz"):
    ev = np.load(EVAL)
    X, y = ev[EVAL_KEY][:N_EVAL], ev["y"][:N_EVAL].astype(int)
else:
    X = np.load(EVAL, mmap_mode="r")[:N_EVAL]
    y = np.load(str(EVAL).replace(".npy", "_y.npy"))[:N_EVAL].astype(int)
Xin = hw_input(X, WINDOW)
print(f"{len(y)} shots, balance {np.bincount(y)}, inputs {Xin.shape[1]}")

## 2. The fixed-point model

The hls4ml configuration must be the one the firmware was built with: these precisions are the deployed ones (see
notebook 03).

In [ ]:
import hls4ml


def make_hls_config(model):
    cfg = hls4ml.utils.config_from_keras_model(model, granularity="name")
    cfg["Model"].update(Precision="ap_fixed<16,6>", ReuseFactor=1, Strategy="Resource")
    L = cfg["LayerName"]

    def prec(layer):
        if not isinstance(L[layer].get("Precision"), dict):
            L[layer]["Precision"] = {}
        return L[layer]["Precision"]

    def result(layer, p):
        if layer in L:
            prec(layer)["result"] = p
            L[layer]["Trace"] = True

    for name in ("input_1", "input_1_input", "q_dense_1_input"):
        if name in L:
            L[name]["Precision"] = "ap_fixed<32,20>"
    result("q_dense_1", "ap_fixed<48,32>"); L["q_dense_1"]["accum_t"] = "ap_fixed<48,32>"
    result("q_output", "ap_fixed<32,16>"); L["q_output"]["accum_t"] = "ap_fixed<32,16>"
    for name in ("batchnorm1", "relu1"):
        result(name, "ap_fixed<32,16>")
    result("sigmoid", "ap_fixed<18,8>")
    for name in list(L):
        if "alpha" in name or "linear" in name:
            result(name, "ap_fixed<24,10>")
    result("q_dense_1_linear", "ap_fixed<48,32>")
    result("q_output_linear", "ap_fixed<32,16>")
    if "batchnorm1" in L:
        prec("batchnorm1")["scale"] = "ap_fixed<40,8>"
        prec("batchnorm1")["bias"] = "ap_fixed<40,8>"
    return cfg

**Windows only.** hls4ml compiles by running `build_lib.sh`, which `cmd.exe` cannot execute. This runs it through MSYS2's
bash instead, with the project path in Windows form (the compiled model loads its weights from that path at run time).
Skip this cell on Linux or macOS.

In [ ]:
if os.name == "nt":
    import subprocess
    from hls4ml.backends.fpga.fpga_backend import FPGABackend

    os.add_dll_directory(str(MSYS_BIN))

    def _compile_with_msys(self, model):
        odir = Path(model.config.get_output_dir())
        sh = odir / "build_lib.sh"
        sh.write_text(sh.read_text().replace('&& pwd)"', '&& pwd -W)"'), newline="\n")
        env = dict(os.environ, PATH=str(MSYS_BIN) + os.pathsep + os.environ["PATH"])
        r = subprocess.run(["bash", "build_lib.sh"], cwd=odir, env=env, text=True,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        if r.returncode != 0:
            print(r.stdout)
            raise RuntimeError("build_lib.sh failed")
        return "{}/firmware/{}-{}.so".format(model.config.get_output_dir(), model.config.get_project_name(),
                                             model.config.get_config_value("Stamp"))

    FPGABackend.compile = _compile_with_msys
    print("hls4ml compile will use", MSYS_BIN)

In [ ]:
prj = OUT_DIR / f"hls4ml_{Path(MODEL_H5).stem}"
hls_model = hls4ml.converters.convert_from_keras_model(model=model, hls_config=make_hls_config(model), output_dir=str(prj),
                                                       part="xczu48dr-ffvg1517-2-e", io_type="io_parallel",
                                                       clock_period=10.0, backend="Vivado", project_name="NN")
hls_model.compile()
print("compiled", prj)

## 3. Float vs fixed point

In [ ]:
def stats(p, y):
    pred = (p > 0.5).astype(int)
    p10, p01 = np.mean(pred[y == 0] == 1), np.mean(pred[y == 1] == 0)
    return 100 * (1 - (p10 + p01) / 2), 100 * p10, 100 * p01


p_float = model.predict(Xin, verbose=0).reshape(-1)
p_fixed = np.asarray(hls_model.predict(Xin)).reshape(-1)

for name, p in (("float (Keras)", p_float), ("fixed (hls4ml)", p_fixed)):
    F, p10, p01 = stats(p, y)
    print(f"{name:15s} fidelity {F:.2f}%   P(1|0) {p10:.2f}%   P(0|1) {p01:.2f}%")
d_float, d_fixed = p_float > 0.5, p_fixed > 0.5
print(f"quantization cost {stats(p_fixed, y)[0] - stats(p_float, y)[0]:+.2f} points; decisions agree on "
      f"{100 * np.mean(d_float == d_fixed):.2f}% of shots")

# Exact (two-sided) McNemar test on the shots where exactly one of the two is right
from scipy.stats import binomtest
right_f, right_x = d_float == y, d_fixed == y
b, c = int(np.sum(right_f & ~right_x)), int(np.sum(~right_f & right_x))
print(f"discordant shots: float right only {b}, fixed right only {c}, exact McNemar p = "
      f"{binomtest(b, b + c).pvalue if b + c else 1.0:.3g}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(p_float, p_fixed, ".", ms=2)
ax[0].plot([0, 1], [0, 1], "k--", lw=0.8)
ax[0].set_xlabel("float probability"); ax[0].set_ylabel("fixed-point probability")
ax[1].hist(p_fixed - p_float, 100)
ax[1].set_xlabel("fixed - float"); ax[1].set_ylabel("shots")
plt.tight_layout(); plt.show()

np.savez(OUT_DIR / "float_vs_fixed.npz", p_float=p_float, p_fixed=p_fixed, y=y)

**Comparing with the hardware.** The fixed-point probabilities above are what the FPGA should output for these inputs,
bit for bit. To check a board, record the classifier's own output for the same input window (the paper's scripts read
NN_live's per-shot record and NN_axi's result BRAM) and compare it with `p_fixed` on the same shots. Any difference
means the input window or the weights differ, not rounding. The fixed-point model has no rounding freedom.

**Notes**
- NN_live is a hand-written streaming HLS core around the same trained weights, not this generated network. Its hardware
  check (`campaign_v140L1/check_live_bitexact.py`) compares the board's per-shot record with a bit-accurate model of that
  core; this notebook covers the generated network that NN_axi uses.
- Try a different window, or narrower types in `make_hls_config`, to see how far the precisions can drop before the
  decisions start to change.